# 15 - Pass 8a: task-adaptive pretraining with MLM, then the usual recipes (one commit, GPU)

Same structure as notebooks 08 and 11-13: fetch the code from GitHub, run the tests, check the data on the CPU,
**pretrain**, then train two runs at a time (one per GPU), then **the analysis runs inside this notebook** so
the result can be judged straight away.

**What Pass 8a asks** (vault: Experiment Plan, "Pass 8 plan"): every run so far starts from the same
off-the-shelf `bert-base-uncased`. None has used the 22,800 training headlines as *text* to adapt BERT
before fine-tuning. The GAN used them badly. Does a plain self-supervised pretraining on them help?
This notebook uses masked-word prediction (MLM), the standard task-adaptive pretraining (TAPT, Gururangan et al. 2020).

| Arm | What it is | Compared with |
|---|---|---|
| A20 | MLM-pretrained encoder, then plain BERT fine-tuning on L (the A1 recipe) | A1 (Pass 1) |
| A21 | MLM-pretrained encoder, then BERT + FGM on L (the A7 recipe, epsilon 0.5) | A7 (Pass 2b) |
| A21 at 100% | the same with every training label | A7 at 100% (Pass 7, 93.90% accuracy) |

5 seeds each, test and masked test sets scored. **Nothing is picked on test:** the pretraining settings,
lr, steps and epsilon are all fixed in advance. Baselines are reused from the committed results
(training is deterministic; every earlier re-run matched exactly).

**Before you run, in the right-hand panel:**
- **Input:** the same `gainsplit-processed` version as notebooks 10-14.
- **Internet: on**, **Accelerator: GPU T4 x2**.
- Then **Save Version -> Save & Run All (Commit)**. About 1.5-2 hours; you can close the browser.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about two minutes)
The repo's tests train a tiny random BERT for a few steps on made-up data. For Pass 8 they also run a
tiny MLM and JEPA pretraining, check that pretraining never reads a label, train the new arms from the
saved encoder, and run the three diffusion arms. They prove the pieces fit together before real GPU
time is spent. If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the data on the CPU (pre-flight)
Checks every file against `manifest.json` and `manifest_pass3.json` (the masked test sets). Any problem raises
an error here, before GPU time is spent. `RUN_FULL100 = True` adds the 5 runs with every label (about 55 of the
notebook's minutes); set it to False only if quota is short.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, preflight
from gainsplit.plans import PLANS

PLAN = "pass8_mlm"
RUN_FULL100 = True
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = (PLANS["pass8_mlm_100"]() if RUN_FULL100 else []) + PLANS[PLAN]()   # longest first

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
display(preflight(configs, DATA_DIR))

## 4. Pretraining on the training headlines, no labels (about 10-20 minutes, GPU 0)
**Masked-word prediction (MLM):** 15% of the words in each headline are hidden and BERT must name them. This is BERT's own pretraining task, now run on our 22,800 training headlines only.

- **Texts:** the `text` column of `headlines_train.csv` only. No sarcasm label is read (a test checks this),
  and dev/test headlines are never trained on (dev texts only give a held-out loss for the curve).
- **Settings (fixed before running, the same for MLM and JEPA):** 10,000 steps, batch 64 (about 28 passes),
  lr 1e-4 with 6% warm-up, max length 48, seed 0. Only the objective differs between notebooks 15 and 16.
- **Linear probe (a quick quality check, before and after):** the encoder is frozen; a logistic regression
  is trained on its vectors for L (1% and 10% labels, seeds 0-4) and scored on dev macro-F1. If pretraining
  made the vectors more useful for sarcasm, the probe goes up. Test is never read.

It runs as a separate process so its GPU memory is freed before training starts. If the session restarts,
this cell skips the pretraining when `pretrain_info.json` already exists.

**What to expect:** dev MLM loss should fall (bert-base starts around 2-3 on news text; headlines are short and odd, so it may start higher).

In [ ]:
import json, shutil
from pathlib import Path
from gainsplit.plans import PASS8_CKPT
CKPT = PASS8_CKPT["mlm"]
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src", CUDA_VISIBLE_DEVICES="0")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
t0 = time.time()
with open(f"{OUT_DIR}/pretrain.log", "w") as log:
    p = subprocess.run([sys.executable, "-m", "gainsplit.ssl_pretrain", "--objective", "mlm",
                        "--data", str(DATA_DIR), "--out", CKPT], stdout=log, stderr=subprocess.STDOUT, env=env)
print(open(f"{OUT_DIR}/pretrain.log").read()[-3000:])
assert p.returncode == 0, "pretraining failed: see pretrain.log above"
shutil.copy(f"{CKPT}/pretrain_info.json", f"{OUT_DIR}/pretrain_info.json")   # receipt goes into the zip
info = json.loads(open(f"{CKPT}/pretrain_info.json").read())
print(f"pretraining: {info['minutes']} min; dev loss {info['dev_before']['loss']} -> {info['dev_after']['loss']}")
for b in ("p1", "p10"):
    print(f"linear probe {b}: dev macro-F1 {info['probe_before'][b]['dev_macro_f1_mean']:.2f} (bert-base) -> "
          f"{info['probe_after'][b]['dev_macro_f1_mean']:.2f} (MLM encoder)")

## 5. The list of runs and the time estimate
25 runs (with `RUN_FULL100`): A21 at 100% x5 first (about 21 min each), then A21 and A20 at 10%
(about 2-3 min each) and at 1% (about 1 min each). The short runs fill the gaps, so **about 1 hour on two GPUs**.

In [ ]:
for c in configs:
    print(f"{c.run_id:16s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} eps={c.fgm_epsilon:g} init={c.init_dir}")
print(len(configs), "runs")

## 6. Run
One run per GPU; when a GPU frees up, the next run starts. Each run's log is in `results/<plan>/logs/<run_id>.log`.
If the session stops halfway, run this cell again: finished runs are skipped.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## 7. Checks and analysis (CPU, about 3 minutes)
1. **Pretraining receipt:** held-out loss and linear probe, before and after.
2. **Means per arm** (points), next to the reused baselines.
3. **Paired bootstrap** over test headlines (10,000 resamples, mean over the 5 seeds), Holm-corrected within
   this notebook's table: exactly the method of Passes 4-7. Contrasts are fixed in
   `src/gainsplit/analysis_pass8.py` (written before any run). `seed_wins` shows on how many seeds the new
   arm is ahead.
4. iSarcasmEval AUC of the new arms minus their baselines (transfer; not expected to move).

In [ ]:
from gainsplit import analysis_pass8 as a8
import pandas as pd
pd.set_option("display.width", 200)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/pass1", f"{REPO_DIR}/results/pass2b", f"{REPO_DIR}/results/pass7", OUT_DIR] if Path(d).exists()]
print("results folders:", RESULTS_DIRS)
ANALYSIS_DIR = f"{OUT_DIR}/analysis"
tables = a8.run_all(RESULTS_DIRS, "mlm", ANALYSIS_DIR, pretrain_infos=[f"{OUT_DIR}/pretrain_info.json"])
for name, t in tables.items():
    print(f"\n=== {name}")
    display(t)

## 8. Verdict against the predictions written before running
Each prediction counts as **supported** only if the difference is positive **and** significant after Holm
(p_holm < 0.05). "Positive but not significant" and "no difference" are reported as they are. The predictions
and their reasoning are in the vault (Experiment Plan, "Pass 8 plan"). Paste this cell's output when you report.

In [ ]:
HYPOTHESES = [('H8a.1', 'A20-A1', 1, 'MLM-TAPT helps plain BERT at 1% labels'), ('H8a.1', 'A20-A1', 10, '... and at 10%'), ('H8a.2', 'A21-A7', 1, 'it stacks with FGM at 1%'), ('H8a.2', 'A21-A7', 10, '... and at 10%'), ('H8a.3', 'A21-A7', 100, 'and still adds with every label')]
f1 = tables["contrasts_f1"].set_index(["contrast", "budget"])
wins = tables["seed_wins"].set_index(["contrast", "budget"])
for h, name, b, text in HYPOTHESES:
    if (name, b) not in f1.index:
        print(f"{h} {name} at {b}%: no runs (skipped)"); continue
    r, w = f1.loc[(name, b)], wins.loc[(name, b)]
    directional = "no direction" not in text
    if directional:
        verdict = ("SUPPORTED" if r.mean_diff > 0 and r.p_holm < 0.05 else
                   "opposite, significant" if r.mean_diff < 0 and r.p_holm < 0.05 else "not supported (no significant difference)")
    else:
        verdict = ("significant difference" if r.p_holm < 0.05 else "no significant difference")
    print(f"{h} {name:24s} {b:3d}%  {r.mean_diff:+.2f} points [{r.ci_low:+.2f}, {r.ci_high:+.2f}]  "
          f"p_holm {r.p_holm:.3f}  seeds ahead {w.seeds_ahead}/{w.n_seeds}  -> {verdict}   ({text})")
if "contrasts_accuracy_100" in tables:
    print("\naccuracy at 100% (the number the seed paper reports):")
    display(tables["contrasts_accuracy_100"])

## Download
Zips the results folder (runs, logs, pretraining receipt, analysis tables). A commit keeps it in the
Output tab: download `results_<plan>.zip` and send it with the notebook log.

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")